In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [ ]:
exports_folder = Path("dp_export_tests")
areas_export_path: Path = exports_folder / "00_add_areas"
clean_db_export_path: Path = exports_folder / "01_cleanup_db"
identify_facets_export_path: Path = exports_folder / "02_identify_facets"

In [3]:
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import DataTirMaps

data_tir_maps: DataFrame = DataTirMaps.bulk_parse(ed).with_columns(
    x_hat=pl.col("latitude").radians().cos() * pl.col("longitude").radians().cos(),
    y_hat=pl.col("latitude").radians().cos() * pl.col("longitude").radians().sin(),
    z_hat=pl.col("latitude").radians().sin(),
).with_columns(
    theta_data_tir_maps = pl.col("z_hat").arccos(),
    phi_data_tir_maps = pl.arctan2(
    pl.col("y_hat"),
    pl.col("x_hat")
    )
)

data_tir_maps

facet_num,latitude,longitude,radius,band depth 350,sigma,band depth 440,slope 1000,ratio 1000,x_hat,y_hat,z_hat,theta_data_tir_maps,phi_data_tir_maps
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,34.076935,128.249664,0.228102,1.00248,0.001575,null,null,null,-0.512783,0.65047,0.560306,0.976042,2.238379
1.0,34.080212,128.760391,0.228082,1.002213,0.001462,null,null,null,-0.518541,0.645848,0.560353,0.975984,2.247293
2.0,34.63451,129.115143,0.227933,1.002295,0.001517,null,null,null,-0.519085,0.638389,0.568339,0.96631,2.253484
3.0,34.572929,129.757385,0.228036,1.002394,0.001397,null,null,null,-0.526599,0.633,0.567455,0.967385,2.264694
4.0,35.244423,130.311279,0.22801,1.002464,0.001438,null,null,null,-0.528354,0.622765,0.577066,0.955665,2.274361
…,…,…,…,…,…,…,…,…,…,…,…,…,…
786427.0,-35.069458,36.039497,0.240551,null,-9999.0,null,null,-9999.0,0.661813,0.481533,-0.574569,2.182874,0.629008
786428.0,-35.012486,36.149452,0.240428,null,-9999.0,null,null,-9999.0,0.661349,0.483139,-0.573755,2.181879,0.630927
786429.0,-35.024361,36.241005,0.240386,null,-9999.0,null,null,-9999.0,0.66048,0.484125,-0.573925,2.182087,0.632525


In [4]:
facets = data_tir_maps.group_by("facet_num").agg(
    pl.col("x_hat").first(),
    pl.col("y_hat").first(),
    pl.col("z_hat").first()
)

facets

facet_num,x_hat,y_hat,z_hat
f64,f64,f64,f64
315110.0,-0.965373,0.254582,0.05695
299243.0,-0.901969,0.430097,0.038321
256893.0,0.52759,-0.794437,-0.300863
465253.0,0.043351,0.956739,-0.287699
444260.0,0.293058,0.911977,-0.287079
…,…,…,…
391708.0,-0.697571,-0.54475,0.465447
748633.0,0.241669,-0.518371,-0.820297
638820.0,0.870672,0.416128,-0.262237


In [ ]:
from typing import List

from tqdm import tqdm

from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from scipy.spatial import cKDTree

closest_facets_dfs : List[DataFrame] = []

facets_xyz = (
        facets
        .select(["x_hat", "y_hat", "z_hat"])
        .to_numpy()
        .astype(np.float32)
    )

for chunk in tqdm(QCubeChunk.generate(2), desc="Running chunks"):
    db_chunk = chunk.filter_lf(pl.scan_parquet(clean_db_export_path)).collect()

    db_chunk_xyz = (
        db_chunk
        .select(["x_hat", "y_hat", "z_hat"])
        .to_numpy()
        .astype(np.float32)
    )

    chunk_tree = cKDTree(db_chunk_xyz, leafsize=32)

    distance, idx = chunk_tree.query(
        facets_xyz,
        k=1,
        workers=-1,
        distance_upper_bound = 0.01, # Verified to contain all still from another higher value run
    )

    db_r = db_chunk["r"].to_numpy()

    # valid nearest neighbours
    valid = idx < len(db_r)

    nearest_r = np.full(len(idx), np.nan)
    nearest_r[valid] = db_r[idx[valid]]

    result = facets.with_columns(
        pl.Series("nearest_r", nearest_r),
        pl.Series("distance", distance)
    )

    closest_facets = result.filter(pl.col("distance") < 0.01)
    
    closest_facets_dfs.append(
        closest_facets
    )


closest_facets_merged = pl.concat(closest_facets_dfs)
closest_facets_merged

Running chunks: 100%|██████████| 96/96 [02:12<00:00,  1.38s/it]


facet_num,x_hat,y_hat,z_hat,nearest_r,distance
f64,f64,f64,f64,f64,f64
137167.0,-0.658846,-0.583159,-0.475234,0.238904,0.053667
134046.0,-0.716976,-0.595051,-0.363124,0.247159,0.086434
164324.0,-0.408257,-0.692252,-0.595074,0.232307,0.000044
663055.0,-0.604497,-0.513587,-0.608943,1.6915e9,0.076163
662531.0,-0.603299,-0.535367,-0.59111,1.6915e9,0.051237
…,…,…,…,…,…
396834.0,0.689842,0.5372,0.485319,0.240558,0.000085
410138.0,0.581327,0.607422,0.541386,0.227377,0.018416
414340.0,0.60473,0.735059,0.306578,0.248677,0.096041


In [10]:
closest_facets_grouped = closest_facets_merged.group_by("facet_num").agg(
    pl.col("distance").min(),
    pl.col("nearest_r").filter(pl.col("distance") == pl.col("distance").min()).first()
)

closest_facets_grouped

facet_num,distance,nearest_r
f64,f64,f64
545359.0,0.000059,0.243067
185823.0,0.000081,0.236677
93794.0,0.000067,0.221631
301039.0,0.000056,0.231488
718850.0,0.000052,0.235359
…,…,…
80098.0,0.000037,0.244022
467329.0,0.000071,0.242154
255150.0,0.000129,0.266316


In [ ]:
data_tir_maps_with_position = data_tir_maps.join(
    closest_facets_grouped,
    "facet_num"
)

data_tir_maps_with_position

In [ ]:
data_tir_maps_with_position.write_parquet(identify_facets_export_path)